# 10. Automated run (execute notebook programmatically)
print('To run this notebook headlessly, use nbconvert or nbclient in a CI environment:')
print('Example: python -m nbclient --execute colab_train_and_full_workflow.ipynb')


In [ ]:
# 9. Save & export outputs
import shutil
Path('outputs').mkdir(exist_ok=True)
df.to_csv('outputs/cleaned.csv', index=False)
import json
summary = {'counts': agg.to_dict(orient='records')}
with open('outputs/summary.json','w') as f:
    json.dump(summary,f)
shutil.make_archive('outputs_bundle','zip','outputs')
print('Saved outputs to outputs_bundle.zip')


In [ ]:
# 8. Visualization of results
import matplotlib.pyplot as plt
agg = aggregate_by_label(df)
plt.bar(agg['label'], agg['count'])
plt.title('Samples per class')
plt.show()


In [ ]:
# 7. Run tests programmatically and summarize
import subprocess
res = subprocess.run(['pytest','-q','tests/test_core.py'], capture_output=True, text=True)
print('pytest stdout:\n', res.stdout)
print('pytest stderr:\n', res.stderr)


In [ ]:
# 6. Unit tests for core functions (pytest)
%%bash
cat > tests/test_core.py <<'PY'
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Any

def aggregate_by_label(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby('label').size().reset_index(name='count')


def test_aggregate_by_label():
    df = pd.DataFrame({'label':['a','b','a']})
    out = aggregate_by_label(df)
    assert out.loc[out['label']=='a','count'].values[0] == 2

PY

pytest -q tests/test_core.py || true


In [ ]:
# 5. Core functions implementation
from typing import Dict

def aggregate_by_label(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate counts by label"""
    return df.groupby('label').size().reset_index(name='count')


def normalize_array(arr: np.ndarray) -> np.ndarray:
    """Normalize numpy array to -1..1"""
    if arr.max() - arr.min() == 0:
        return arr
    return (arr - np.mean(arr)) / (np.std(arr) + 1e-6)

print('Core functions defined')


In [ ]:
# 4. Data cleaning & transformation
import pandas as pd
# Demonstrate loading created WAV list into a DataFrame
from glob import glob
rows = []
for cls in ['wake','notwake']:
    for p in glob(f'sample_data/{cls}/*.wav'):
        rows.append({'path':p, 'label':cls})
df = pd.DataFrame(rows)
print('Before:')
print(df.head())
# No missing values in synthetic set, but show transformation
df['duration_s'] = 1.0
print('After:')
print(df.info())


In [ ]:
# 3. Create sample dataset (synthetic example)
from pathlib import Path
DATA_DIR = Path('sample_data')
wake_dir = DATA_DIR / 'wake'
notwake_dir = DATA_DIR / 'notwake'
wake_dir.mkdir(parents=True, exist_ok=True)
notwake_dir.mkdir(parents=True, exist_ok=True)

# Create synthetic sine + noise 'wake' and random noise 'notwake' WAVs for demo
import soundfile as sf
sr = 16000
for i in range(10):
    t = np.linspace(0,1,sr)
    tone = 0.1 * np.sin(2*np.pi*300*t) + 0.02*np.random.randn(sr)
    sf.write(str(wake_dir/f'wake_{i}.wav'), tone.astype(np.float32), sr)
for i in range(20):
    noise = 0.02*np.random.randn(sr)
    sf.write(str(notwake_dir/f'not_{i}.wav'), noise.astype(np.float32), sr)

print('Synthetic dataset created at', DATA_DIR)


In [ ]:
# 2. VS Code Integration Check (simulated)
# List current working directory files and run a simple shell command
import os
print('CWD:', os.getcwd())
print('Files:', os.listdir('.'))
# Run a simple shell command
proc = subprocess.run(['echo', 'VSCode simulated terminal command'], capture_output=True, text=True)
print('Terminal output:', proc.stdout)


In [ ]:
# 1. Install and import dependencies

# Run this cell in Colab or local environment. Colab: prefix pip with !
!pip install -q tensorflow soundfile librosa nbformat nbclient pytest matplotlib pandas

import os
import sys
import pathlib
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print('Environment ready')


# KWS Training & Workflow Notebook
This notebook demonstrates a full "do all" workflow for training a wake-word ("hey imi") model, verifying integration, generating artifacts, running unit tests, and exporting outputs. It's Colab-friendly and can also be run locally.
